# Optimizing LibreYOLO for Qualcomm® Snapdragon Devices Using QAIRT

This notebook walks through the full pipeline to optimize a **LibreYOLO** object detection model for efficient inference on Qualcomm® Snapdragon devices using the **Qualcomm AI Runtime (QAIRT) SDK**.

The pipeline covers:
1. **Image preprocessing** — resize and normalize images for model input.
2. **Calibration dataset preparation** — generate representative `.raw` samples for INT8 quantization.
3. **Model export** — load a pre-trained LibreYOLOXs checkpoint and export it to ONNX.
4. **DLC conversion** — convert the ONNX model to QAIRT's Deep Learning Container (DLC) format using `qairt-converter`.
5. **INT8 quantization** — apply post-training quantization for faster, more efficient inference using `qairt-quantizer`.
6. **HTP context binary compilation** — compile the DLC offline for the Hexagon Tensor Processor (HTP) on the Snapdragon 778G (sm7325) using `qairt-context-binary-generator`.

We begin by importing the necessary libraries.

In [1]:
# Import necessary libraries.
import glob
import os
import onnx
import random
import torch
import uuid

import numpy as np

from libreyolo.models.yolox.nn import LibreYOLOXModel
from PIL import Image

## Preprocessing and Calibration Data

Before converting or quantizing the model, two things are needed:

- A **preprocessing function** to transform raw images into the tensor format expected by LibreYOLO (resized to 640×640, normalized to `[0, 1]`).
- A **representative calibration dataset** in `.raw` binary format. QAIRT's `qairt-quantizer` uses this dataset during post-training quantization to compute per-layer scale factors for INT8 weights and activations.

The cell below defines the `preprocess()` helper function used throughout this pipeline.

In [2]:
def preprocess(original_image: np.ndarray, size: int = 640) -> np.ndarray:
    """
    Preprocess the input image for model inference.

    Args:
        original_image (np.ndarray): Input image as NumPy array.
        size (int): Target image size.

    Returns:
        np.ndarray: Preprocessed float32 image normalized to [0, 1].
    """

    image = Image.fromarray(original_image).convert("RGB")
    image = image.resize((size, size), Image.BILINEAR)

    image = np.asarray(image).astype(np.float32) / 255.0
    image = np.transpose(image, (2, 0, 1))  # HWC -> CHW
    image = np.expand_dims(image, axis=0)   # CHW -> NCHW

    return image

### Preparing the Calibration Dataset

QAIRT's quantization tool (`qairt-quantizer`) requires input samples in `.raw` format — flat binary files containing `float32` pixel values with shape `(H, W, C)`.

The code below:
1. Downloads the **COCO 2017 validation set** (~777 MB).
2. Randomly samples **1000 images** (with a fixed seed for reproducibility).
3. Preprocesses each image using `preprocess()` and saves it as a `.raw` file.
4. Generates a `filenames.txt` manifest listing all `.raw` files — required by `qairt-quantizer`.

In [3]:
if not os.path.exists("val"):
    !bash -c 'curl -L -o val2017.zip http://images.cocodataset.org/zips/val2017.zip'
    !bash -c 'unzip val2017.zip -d val'
    !bash -c 'rm val2017.zip'

if not os.path.exists("raw"):
    !bash -c 'mkdir raw'

    random.seed(42)
    SAMPLE_SIZE = 1000

    filenames = glob.glob(f"val/**/*.jpg", recursive=True)
    random.shuffle(filenames)

    for filename in filenames[:SAMPLE_SIZE]:
        image = Image.open(filename)
        original_image = np.array(image)
        processed_image = preprocess(original_image)
        processed_image.tofile(f"raw/{uuid.uuid4()}.raw")

if not os.path.exists("raw/filenames.txt"):
    !bash -c 'find raw -name "*.raw" > raw/filenames.txt'

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  777M  100  777M    0     0  17.2M      0  0:00:45  0:00:45 --:--:-- 18.0M
Archive:  val2017.zip
   creating: val/val2017/
 extracting: val/val2017/000000212226.jpg  
 extracting: val/val2017/000000231527.jpg  
 extracting: val/val2017/000000578922.jpg  
 extracting: val/val2017/000000062808.jpg  
 extracting: val/val2017/000000119038.jpg  
 extracting: val/val2017/000000114871.jpg  
 extracting: val/val2017/000000463918.jpg  
 extracting: val/val2017/000000365745.jpg  
 extracting: val/val2017/000000320425.jpg  
 extracting: val/val2017/000000481404.jpg  
 extracting: val/val2017/000000314294.jpg  
 extracting: val/val2017/000000335328.jpg  
 extracting: val/val2017/000000513688.jpg  
 extracting: val/val2017/000000158548.jpg  
 extracting: val/val2017/000000132116.jpg  
 extracting: val/val2017/000000415238.jpg  
 extracting

### Loading LibreYOLO and Exporting to ONNX

SNPE does not consume LibreYOLO models in PyTorch format directly. The model must first be exported to the **ONNX (Open Neural Network Exchange)** format, which SNPE can then parse and convert to its DLC format.

The code below:
1. Downloads the pre-trained **LibreYOLOXs** checkpoint from Hugging Face.
2. Loads it using the `LibreYOLO` wrapper and sets it to evaluation mode.
3. Exports it to ONNX using `torch.onnx.export()` with `opset_version=18`.

The model decodes the grid offsets inside the graph and returns:
[1, 8400, 85]

where:

8400 = 80x80 + 40x40 + 20x20
85 = 4 bbox values + 1 objectness + 80 class probabilities

The bbox format is:

[cx, cy, w, h]

in the resized 640x640 input coordinate space. When use the model, it is necessary the confidence filtering, `cxcywh -> xyxy` conversion, scaling back to the original image, and NMS.

In [4]:
os.makedirs("../models", exist_ok=True)
os.makedirs("../models/qairt", exist_ok=True)

if not os.path.exists("../models/LibreYOLOXs.pt"):
    !bash -c 'curl -L -o ../models/LibreYOLOXs.pt https://huggingface.co/LibreYOLO/LibreYOLOXs/resolve/main/LibreYOLOXs.pt?download=true'

checkpoint = torch.load(
    "../models/LibreYOLOXs.pt",
    map_location="cpu"
)

model = LibreYOLOXModel(config="s", nb_classes=80)
model.load_state_dict(checkpoint["model"], strict=True)

model.eval()
model.head.export = True

dummy = torch.randn(1, 3, 640, 640)
onnx_path = "../models/LibreYOLOXs.onnx"

if not os.path.exists(onnx_path):
    torch.onnx.export(
        model,
        dummy,
        onnx_path,
        opset_version=13,
        dynamo=False,
        do_constant_folding=True,
        input_names=["images"],
        output_names=["detections"]
    )

    print(f"Exported decoded ONNX to: {onnx_path}.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1349  100  1349    0     0   4271      0 --:--:-- --:--:-- --:--:--  4282
100 68.7M  100 68.7M    0     0  36.7M      0  0:00:01  0:00:01 --:--:-- 56.9M


/tmp/ipykernel_80/605095049.py:22: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Exported decoded ONNX to: ../models/LibreYOLOXs.onnx.


### Validating the ONNX output shape

This cell checks that the exported ONNX has exactly one output named `detections` with shape `[1, 8400, 85]`. If you still see three outputs, the model was exported without `torch_model.head.export = True`.

In [5]:
onnx_model = onnx.load("../models/LibreYOLOXs.onnx")
onnx.checker.check_model(onnx_model)

print("Inputs:")
for tensor in onnx_model.graph.input:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    print(f" {tensor.name}: {dims}")

print("Outputs:")
output_shapes = {}
for tensor in onnx_model.graph.output:
    dims = [d.dim_value if d.dim_value > 0 else d.dim_param for d in tensor.type.tensor_type.shape.dim]
    output_shapes[tensor.name] = dims
    print(f" {tensor.name}: {dims}")

assert len(onnx_model.graph.output) == 1, "Expected exactly one decoded output."
assert "detections" in output_shapes, "Expected output named `detections`."
assert output_shapes["detections"] == [1, 8400, 85], f"Unexpected output shape: {output_shapes['detections']}"

print("ONNX export is correct: one decoded output [1, 8400, 85].")

Inputs:
 images: [1, 3, 640, 640]
Outputs:
 detections: [1, 8400, 85]
ONNX export is correct: one decoded output [1, 8400, 85].


## Converting the Model to QAIRT DLC Format

QAIRT uses the **DLC (Deep Learning Container)** format as an intermediate representation. The first step is to convert the ONNX model to a floating-point (**FP32**) DLC using the `qairt-converter` tool. This DLC file is the starting point for all subsequent quantization and compilation steps.

In [6]:
!bash -c 'qairt-converter \
    --input_network ../models/LibreYOLOXs.onnx \
    --output_path ../models/qairt/LibreYOLOXs_fp32.dlc'

2026-05-10 10:02:47,685 - 278 - INFO - INFO_INITIALIZATION_SUCCESS: 
2026-05-10 10:02:47,739 - 283 - WARNING - Unable to register converter supported Operation [Inverse:Version 1] with your Onnx installation. Converter will bail if Model contains this Op.
2026-05-10 10:02:47,825 - 283 - WARNING - --desired_input_shape and -d are deprecated. Use --source_model_input_shape or -s for achieving this functionality
2026-05-10 10:02:47,826 - 278 - INFO - Input shape info 
2026-05-10 10:02:49,632 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-10 10:02:49,633 - 283 - WARNING - Simplified model validation failed
2026-05-10 10:02:49,846 - 283 - WARNING - Onnx model simplification failed due to: No module named 'onnxsim'
2026-05-10 10:02:49,846 - 283 - WARNING - Simplified model validation failed
2026-05-10 10:02:50,004 - 278 - INFO - INFO_STATIC_RESHAPE: Applying static reshape to /head/Concat_3_output_0: new name /head/Reshape_1_output_0 new shape [1

### Inspecting the FP32 DLC

The `qairt-dlc-info` command prints a detailed summary of the DLC graph, including layer names, types, input/output tensor shapes, and supported execution backends (CPU, GPU, HTP). Reviewing this output verifies that the ONNX export was clean and that all operators are supported by QAIRT before proceeding to quantization.

In [7]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc'

DLC info of: /workspace/models/qairt/LibreYOLOXs_fp32.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: qairt-converter; validate_models=False; use_quantize_v2=False; use_onnx_relay=False; unroll_lstm_time_steps=True; unroll_gru_time_steps=True; transforms_metadata=None; tf_summary=False; skip_validation=False; signature_name=; dump_value_info=False; remove_unused_inputs=False; expand_sparse_op_structure=False; quantizer_log_level=LogLevel.NONE; quantization_overrides=; enable_match_topk=False; pytorch_custom_op_lib=; prepare_inputs_as_params=False; perform_axes_to_spatial_first_order=False; define_symbol=None; use_convert_quantization_nodes=False; packed_max_seq=1; packed_masked_softmax_inputs=[]; output_layout=[]; float_bias_bitwidth=0; show_unconsumed_nodes=False; out_names=[]; float_bitwidth=32; optimization_pass_mode=ir_optimizer_disable; apply_masked_softmax=uncompressed; en

### INT8 Post-Training Quantization

Running inference in **8-bit integer (INT8)** precision is significantly faster and more energy-efficient on Qualcomm® hardware than FP32 — with only a small accuracy trade-off. The `qairt-quantizer` tool applies **post-training quantization (PTQ)** by computing per-layer scale factors from the calibration `.raw` samples prepared earlier, producing an INT8 DLC.

In [8]:
!bash -c 'qairt-quantizer \
    --input_dlc ../models/qairt/LibreYOLOXs_fp32.dlc \
    --input_list raw/filenames.txt \
    --output_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

     2.0ms [  INFO ] Inferences will run in sync mode
Processing inference input(s):
raw/7f3d6c81-4ee7-4e5a-9db5-0a4fbde7f5a3.raw
raw/df20eefc-43d2-4a33-8195-336f13c7a94f.raw
raw/c5facc87-f986-4144-8ddd-056e65663254.raw
raw/8928b015-c7da-4ead-9fa7-f2b3524d6562.raw
raw/b878b7bc-783c-42c6-93a5-af96a6f13229.raw
raw/4e4caa8e-933e-46a3-abb5-c7e652b29284.raw
raw/554ab847-3533-44c6-9efa-d764f1b8bc91.raw
raw/086e39ed-9abb-4b7e-9623-0c11002a1257.raw
raw/4bca67a3-f517-4534-a300-69a2a68c2c94.raw
raw/2ca4ba51-5a2c-4911-89e8-36ebb00a9729.raw
raw/327570f1-ab49-4672-9aa7-05130821120e.raw
raw/a137e2f2-54a5-4902-9864-854dabc730a9.raw
raw/779ae593-f5d8-4bd4-8d93-ea0260917fd1.raw
raw/3540cb8d-1f0c-44d9-a456-d61f44572d74.raw
raw/a844bf4e-0e09-44ae-a2ad-225c44557a52.raw
raw/1dd1773e-60f5-44f6-8bf8-54e18ea54644.raw
raw/1394c28b-020f-4dd0-bd54-6f49f7e48c43.raw
raw/5446f0e0-7e02-429c-ad9b-326f3f193060.raw
raw/bbd71cbc-7c5d-4676-a125-048ad48d397a.raw
raw/1fccf14b-6d02-4585-be56-7cb31b938934.raw
raw/7532f81b-e7

### Inspecting the INT8 DLC

After quantization, `qairt-dlc-info` is used again to inspect the INT8 DLC. Compared to the FP32 output, you should observe that layer types now reflect quantized variants. The INT8 DLC is expected to be smaller than the FP32 DLC, but the actual reduction should be measured.

In [9]:
!bash -c 'qairt-dlc-info \
    --input_dlc ../models/qairt/LibreYOLOXs_int8.dlc'

DLC info of: /workspace/models/qairt/LibreYOLOXs_int8.dlc
                                                                                                    
Model Version: 
Model Copyright:
Converter command: qairt-converter; optimization_pass_mode_config=; batch=None; squash_box_decoder=True; dump_exported_onnx=False; dumpIR=False; custom_op_config_paths=None; preprocess_roi_pool_inputs=True; saved_model_tag=serve; backend=HTP; preserve_io=[['layout']]; multi_time_steps_lstm=False; debug=-1; calc_static_encodings=False; dump_custom_io_config_template=; partial_graph_input_name=None; perform_sequence_construct_optimizer=False; adjust_nms_features_dims=True; user_custom_io=[]; disable_transform_tracking=False; disable_defer_loading=False; enable_Layout_Transform_v1=False; inject_cast_for_gather=True; converter_op_package_lib=; enable_experimental_feature=[]; handle_gather_negative_indices=True; disable_preserve_io=False; enable_tensor_deduplication=False; disable_batchnorm_folding=Fal

By following these steps, the model is optimized for efficient inference on Qualcomm® platforms, leveraging hardware acceleration for real-time AI applications. This process ensures that the model is both accurate and performant, making it suitable for deployment in edge devices powered by Qualcomm® chipsets.